# 06 - Monte Carlo portfolio loss (single-factor Vasicek)

Loss is simulated loan-by-loan through homogeneous buckets (`grade` x PD
decile). Conditional on the systematic factor `Z`,

$$p_i(Z) = \Phi\!\left(\frac{\Phi^{-1}(PD_i) - \sqrt{\rho}\,Z}{\sqrt{1-\rho}}\right),\qquad
L = \sum_i EAD_i\,LGD_i\,D_i,$$

with `D_i ~ Bernoulli(p_i(Z))`. We report Expected Loss (EL), Unexpected Loss
(UL), Value-at-Risk at 99% / 99.9%, Expected Shortfall, a convergence study
(Monte Carlo error decays like `1/sqrt(N)`), antithetic variates as a
variance-reduction technique, and a stress-test grid.

In [ ]:
import json, pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / 'config' / 'config.yaml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 180)
RES = ROOT / 'reports' / 'results'

## Rebuild the simulated portfolio

`python -m src.pipeline simulation` persists the bucket table that came out of
the out-of-time PD and LGD models. We reload it and re-run the simulation here.

In [ ]:
from src import simulation as sim

buckets = pd.read_csv(RES / 'sim_portfolio_buckets.csv')
display(buckets.head(10).round(4))
portfolio = sim.BucketedPortfolio(
    ead=buckets['ead'].to_numpy(dtype=float),
    lgd=buckets['lgd'].to_numpy(dtype=float),
    pd=buckets['pd'].to_numpy(dtype=float),
    n=buckets['n_loans'].to_numpy(dtype=float),
    labels=list(buckets['bucket'].astype(str)),
)
print('buckets:', len(portfolio.labels), ' total EAD:', round(portfolio.total_ead, 2))

In [ ]:
cfg = json.loads((RES / 'summary.json').read_text()) if (RES / 'summary.json').exists() else {}
rho = float(cfg.get('simulation', {}).get('asset_correlation', 0.12))
from src.simulation import simulate_losses
run = simulate_losses(portfolio, rho=rho, n_simulations=100_000, seed=42, antithetic=True)
display(pd.DataFrame([run['stats']]).T.rename(columns={0: 'value'}).round(6))

`var_0.99` and `var_0.999` are portfolio Value-at-Risk (the loss level exceeded
with probability 1% and 0.1% respectively). `economic_capital_0.999 = VaR(99.9%) - EL`
is the classic unexpected-loss buffer.

In [ ]:
from src.simulation import convergence_study, variance_reduction_report, stress_test
rep = variance_reduction_report(portfolio, rho=rho, n_max=100_000, seed=42)
display(rep.round(2))

In [ ]:
stress = stress_test(portfolio, pd_shocks=(0.30, 0.50), lgd_shocks=(0.10,), seed=42)
display(stress[['scenario', 'expected_loss', 'unexpected_loss', 'var_0.99', 'var_0.999', 'es_0.99']].round(0))

In [ ]:
for name in ['mc_loss_distribution.png', 'mc_convergence.png', 'mc_stress_test.png']:
    p = ROOT / 'reports' / 'figures' / name
    if p.exists():
        display(Image(filename=str(p)))

### Assumptions to state explicitly

* **Asset correlation** `rho` is an assumption (default 0.12), not an estimate;
  retail portfolios typically map to `rho` in the 0.03-0.16 range under the
  Basel IRB formula. Its effect on VaR is second only to the PD level.
* **LGD is fixed per bucket** at its predicted mean; stochastic LGD (e.g. a
  Beta draw correlated with the systematic factor) would fatten the tail.
* Exposure is the **funded amount** with no amortisation, so losses are
  slightly conservative relative to a true exposure-at-default profile.
* The portfolio is the **out-of-time book** (2017-2018 vintages), scored with
  the champion PD model and the LGD model from notebooks 03 and 04.